In [13]:
SILVER_PATH = "Files/silver/retail_transactions"
 
GOLD_DB = "RetailLakehouse"


StatementMeta(, dd344abe-3d6c-4a1f-a3c3-2d9b5b62f033, 15, Finished, Available, Finished, False)

In [14]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
df_silver = spark.read.format("delta").load(SILVER_PATH)
print(f"Silver rows loaded: {df_silver.count()}")


StatementMeta(, dd344abe-3d6c-4a1f-a3c3-2d9b5b62f033, 16, Finished, Available, Finished, False)

Silver rows loaded: 757038


In [18]:
dim_customer = (df_silver
    .select("Customer_ID", "Country")
    .dropDuplicates(["Customer_ID"])
    .withColumn("customer_key",
        F.row_number().over(Window.orderBy("Customer_ID"))))

 
print(f"dim_customer: {dim_customer.count()} rows")


StatementMeta(, dd344abe-3d6c-4a1f-a3c3-2d9b5b62f033, 20, Finished, Available, Finished, False)

dim_customer: 5860 rows


In [19]:
dim_product = (df_silver
    .select("StockCode", "Description")
    .dropDuplicates(["StockCode"])
    .withColumn("product_key",
        F.row_number().over(Window.orderBy("StockCode"))))
 
print(f"dim_product: {dim_product.count()} rows")


StatementMeta(, dd344abe-3d6c-4a1f-a3c3-2d9b5b62f033, 21, Finished, Available, Finished, False)

dim_product: 4630 rows


In [28]:
dim_date = (df_silver
    .select(F.to_date("InvoiceDate", "dd-MM-yyyy HH:mm").alias("date"))
    .dropDuplicates()
    .withColumn("date_key",
        F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year",       F.year("date"))
    .withColumn("month",      F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("quarter",    F.quarter("date"))
    .withColumn("day",        F.dayofmonth("date"))
    .withColumn("weekday",    F.dayofweek("date")))
 
print(f"dim_date: {dim_date.count()} rows")


StatementMeta(, dd344abe-3d6c-4a1f-a3c3-2d9b5b62f033, 30, Finished, Available, Finished, False)

dim_date: 599 rows


In [23]:
fact_sales = (df_silver
    .join(dim_customer.select("Customer_ID", "customer_key"),
          on="Customer_ID", how="left")
    .join(dim_product.select("StockCode", "product_key"),
          on="StockCode", how="left")
    .join(dim_date.select("date", "date_key"),
          on=F.to_date(df_silver["InvoiceDate"]) == F.col("date"),
          how="left")
    .select(
        F.col("Invoice").alias("invoice_no"),
        F.col("customer_key"),
        F.col("product_key"),
        F.col("date_key"),
        F.col("Quantity").alias("quantity"),
        F.col("Price").alias("unit_price"),
        F.col("LineTotal").alias("line_total")))
 
print(f"fact_sales: {fact_sales.count()} rows")

StatementMeta(, dd344abe-3d6c-4a1f-a3c3-2d9b5b62f033, 25, Finished, Available, Finished, False)

fact_sales: 757038 rows


In [29]:
tables = {
    "dim_customer": dim_customer,
    "dim_product":  dim_product,
    "dim_date":     dim_date,
    "fact_sales":   fact_sales
}
 
for name, df in tables.items():
    df.write.mode("overwrite").format("delta").saveAsTable(name)
    print(f"Written Gold table: {name} ({df.count()} rows)")
 
print("Silver → Gold complete. Star schema ready in Lakehouse Tables.")


StatementMeta(, dd344abe-3d6c-4a1f-a3c3-2d9b5b62f033, 31, Finished, Available, Finished, False)

Written Gold table: dim_customer (5860 rows)
Written Gold table: dim_product (4630 rows)
Written Gold table: dim_date (599 rows)
Written Gold table: fact_sales (757038 rows)
Silver → Gold complete. Star schema ready in Lakehouse Tables.
